In [1]:
import os
import sys
import glob
import pickle
import shutil
import zipfile

import numpy as np
import matplotlib.pyplot as plt

### FILES ###
import shutil # delate directory
import zipfile # extract a directory

In [2]:
# DELETE A DIRECTORY FROM CONTENT

folder = "/content/Dataset"

if os.path.exists(folder):
    shutil.rmtree(folder)
    print("Delated directory")
else:
    print("No delated directory")

Delated directory


In [3]:
# EXTRACT ZIP DIRECTORIES

from google.colab import drive
drive.mount('/content/drive')

LOCAL_ROOT = "/content/Dataset"

def unzip_file(zip_path):

    os.makedirs(LOCAL_ROOT, exist_ok=True)

    zip_name = os.path.basename(zip_path)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(LOCAL_ROOT)

    print(f"{zip_name} has extracted.")

#unzip_file("/content/drive/MyDrive/DATASET_SHARP/Python_code_old.zip")
unzip_file("/content/drive/MyDrive/DATASET_SHARP/doppler_traces.zip")
#unzip_file("/content/drive/MyDrive/DATASET_SHARP/doppler_traces_S4_S5.zip")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
doppler_traces.zip has extracted.


In [4]:
# RETURN THE ACTIONS FOR EACH DIRECTORY

def getActions(folder_path):

    folder_name = os.path.basename(os.path.normpath(folder_path))

    actions = []

    for filename in os.listdir(folder_path):

        prefix = f"{folder_name}_"
        marker = "_stream_"

        if not filename.startswith(prefix):
            continue

        if marker not in filename:
            continue

        remaining = filename[len(prefix):]

        action = remaining.split(marker)[0]

        if action not in actions:
            actions.append(action)

    actions.sort()

    print("Actions:", actions)

    return actions

In [5]:
# LISTA DELLE CARTELLE DEL DATASET

ROOT_PATH = "/content/Dataset/doppler_traces"

folders = []

for folder_name in sorted(os.listdir(ROOT_PATH)):

    folder_path = os.path.join(ROOT_PATH, folder_name)

    # Considera solo le cartelle
    if not os.path.isdir(folder_path):
        continue

    folders.append(folder_path)

print(folders)


['/content/Dataset/doppler_traces/S1a', '/content/Dataset/doppler_traces/S1b', '/content/Dataset/doppler_traces/S1c', '/content/Dataset/doppler_traces/S2a', '/content/Dataset/doppler_traces/S2b', '/content/Dataset/doppler_traces/S3a', '/content/Dataset/doppler_traces/S4a', '/content/Dataset/doppler_traces/S4b', '/content/Dataset/doppler_traces/S5a', '/content/Dataset/doppler_traces/S6a', '/content/Dataset/doppler_traces/S6b', '/content/Dataset/doppler_traces/S7a']


In [6]:
# EXTRACT DATASET

complete_dataset = {}

for i in range(0, len(folders)):

  folder = folders[i]
  #print(folder)
  actions = getActions(folder)
  dataset_array = {}

  for action in actions:
    dataset_array[action] = []

  #print("Dataset array: ", dataset_array)

  folder_name = os.path.basename(folder)
  #print("Folder name: ", folder_name)

  for action in actions:

    for filename in os.listdir(folder):
      if filename.startswith(f"{folder_name}_{action}"):

        file_path = os.path.join(folder, filename)

        with open(file_path, "rb") as fp:
          doppler = pickle.load(fp)

        dataset_array[action].append(doppler)

  complete_dataset[folder_name] = dataset_array

  # ** PRINT **
  #print("\n==========================")
  #print("Cartella:", folder_name)

  #for action in dataset_array:

    #print(f"\nAzione: {action}")
    #print("Numero stream:", len(dataset_array[action]))

    #for i, stream in enumerate(dataset_array[action]):
        #print(f"  Stream {i}: shape {np.array(stream).shape}")

Actions: ['C', 'E', 'H', 'J1', 'J2', 'L', 'R', 'S', 'W']
Actions: ['C', 'E', 'H', 'J1', 'J2', 'L', 'R', 'S', 'W']
Actions: ['C', 'E', 'H', 'J1', 'J2', 'L', 'R', 'S', 'W']
Actions: ['E', 'J', 'L', 'R', 'W']
Actions: ['E', 'J1', 'J2', 'L', 'R', 'W']
Actions: ['E', 'H', 'J1', 'J2', 'L', 'R', 'S', 'W']
Actions: ['C1', 'C2', 'E', 'H1', 'H2', 'J1', 'J2', 'L', 'R', 'S', 'W']
Actions: ['E', 'J1', 'J2', 'L', 'R', 'W']
Actions: ['C1', 'C2', 'E', 'H1', 'H2', 'J1', 'J2', 'L', 'LOS', 'R', 'W']
Actions: ['C', 'E', 'H', 'J1', 'J2', 'L', 'R', 'S', 'W']
Actions: ['E', 'J1', 'J_0', 'L', 'R', 'W']
Actions: ['C', 'E', 'H', 'J1', 'J2', 'J3', 'L1', 'L2', 'R1', 'S', 'W']


In [7]:
# WINDOW OF 1@(340 x 100)

def create_sliding_windows(complete_dataset, window_length=340, stride=340):

  X = []
  y = []
  folders = []

  for folder_name in complete_dataset:
    print("Cartella: ", folder_name)
    dataset = complete_dataset[folder_name]
    all_windows = {}

    for action in dataset:

      data = np.asarray(dataset[action])
      windows_activity = []
      # elements of each action
      num_streams, time_length, num_features = np.array(data).shape
      print(f"Action {action} -> Shape of data: {num_streams}, {time_length}, {num_features}")

      for stream in range(0, num_streams):

        window = []
        # se lo stream è più corto in partenza della grandezza della finestra
        if time_length < window_length:
          print("The dataset is less than window size")
          continue

        start = 0
        end = 340
        while end <= time_length:
          window = data[stream][start:end,:]
          #windows_stream.append(window)
          X.append(window)
          y.append(action)
          folders.append(folder_name)
          start += stride
          end += stride

        #if start < time_length and end > time_length:
          #last_window = np.array(data[stream][start:time_length,:])
          #print("Zero da aggiungere: ", window_length - last_window.shape[0])
          #zero_padding = np.zeros((window_length - last_window.shape[0], last_window.shape[1]), dtype=last_window.dtype)
          #last_window = np.vstack((last_window, zero_padding))
          #windows_stream.append(last_window)
          #print("Shape ultima finestra: ", last_window.shape)
          #print("NUmbers of zeros: ", len(zero_padding))
          #print("last window: ", last_window)

      del data

    del dataset

  return X, y, folders

X, y, folders = create_sliding_windows(complete_dataset)

#print("\n==========================")
#print("DATASET FINALE")
#print("==========================")

#for folder_name in complete_dataset:

    #print(f"\nCartella: {folder_name}")

    #for action in complete_dataset[folder_name]:

        #print(
            #f"  {action}: {complete_dataset[folder_name][action].shape}"
        #)


Cartella:  S1a
Action C -> Shape of data: 4, 18766, 100
Action E -> Shape of data: 4, 18700, 100
Action H -> Shape of data: 4, 19064, 100
Action J1 -> Shape of data: 4, 8700, 100
Action J2 -> Shape of data: 4, 8708, 100
Action L -> Shape of data: 4, 18842, 100
Action R -> Shape of data: 4, 19269, 100
Action S -> Shape of data: 4, 19009, 100
Action W -> Shape of data: 4, 19295, 100
Cartella:  S1b
Action C -> Shape of data: 4, 18803, 100
Action E -> Shape of data: 4, 18270, 100
Action H -> Shape of data: 4, 18707, 100
Action J1 -> Shape of data: 4, 8756, 100
Action J2 -> Shape of data: 4, 8667, 100
Action L -> Shape of data: 4, 19329, 100
Action R -> Shape of data: 4, 19253, 100
Action S -> Shape of data: 4, 18922, 100
Action W -> Shape of data: 4, 18927, 100
Cartella:  S1c
Action C -> Shape of data: 4, 18533, 100
Action E -> Shape of data: 4, 18929, 100
Action H -> Shape of data: 4, 19053, 100
Action J1 -> Shape of data: 4, 8279, 100
Action J2 -> Shape of data: 4, 8784, 100
Action L -> 

In [8]:
index = np.random.randint(0, len(X))
print("Index: ", index)

print("Element in X: ", X[index].shape)
print("Element in y: ", y[index])

Index:  6690
Element in X:  (340, 100)
Element in y:  R


In [ ]:
# TRAINING, VALIDATION E TEST